---
<h1 align="center"><b>PIPELINE en Análitica de Datos</b></h1> 

---

- Un Pipeline en Python (específicamente en el ecosistema de *Scikit-Learn*) es un objeto que encadena múltiples pasos de procesamiento de datos y un estimador final en un solo flujo de trabajo.

- Funciona tipo línea de ensamblaje, los datos crudos entran por un extremo, pasan por varias máquinas (limpieza, escalado, selección de variables) y salen por el otro extremo como una predicción final.

---

<h1 align="center">Datos → Limpieza → Transformación → Modelo → Predicción</h1> 

---

<p align="center">
   <img src="img/pipe.png" width="900" height="500">
</p>

---

## ¿Por qué es importante?

✔️ Evita Data Leakage (fuga de datos)

✔️ Automatiza el flujo completo

✔️ Facilita validación cruzada

✔️ Es estándar en industria

✔️ Permite reproducibilidad

---

## Estructura de un Pipeline

Un Pipeline consta típicamente de dos tipos de componentes:

- **Transformers:** Pasos intermedios que limpian o transforman los datos, por ejemplo: 

*(StandardScaler, SimpleImputer)*. Deben tener los métodos *.fit() y .transform()*.

- **Estimator:** El paso final, que suele ser el algoritmo de Machine Learning, por ejemplo: 

*(RandomForest, LinearRegression)*. Debe tener el método *.fit().*

---

### ❌ MAL enfoque

```
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model.fit(X_train_scaled, y_train)
```

---

### ✅ TODO integrado correctamente

```
Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
```
---

### 🧪 Pipeline de Principio a Fin

```
# Pipeline para numéricas
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para categóricas
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Modelo
model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', LinearRegression())
])
```

---

---
<h1 align="center"><b>Variables Categórica y Codificación</b></h1> 

---

En muchos modelos estadísticos y de aprendizaje automático, las variables categóricas deben transformarse en variables numéricas para poder ser utilizadas.

---

<p align="center">
 <img src="img/cat_codi.png" width="500" height="700">
</p>

---


### **Variables Dummy**

Sea una variable categórica $X$ con $k$ categorías. La codificación *Dummy* consiste en crear $k-1$ variables binarias:

$$
D_j =
\begin{cases}
1 & \text{si } X = \text{categoría } j \\
0 & \text{en otro caso}
\end{cases}
$$

para $j = 1, \dots, k-1$.

Esto evita la multicolinealidad perfecta, conocida como la \textbf{trampa de las variables dummy}.

---

### **One-Hot Encoding**

El One-Hot Encoding transforma una variable categórica con $k$ categorías en $k$ variables binarias:

$$
O_j =
\begin{cases}
1 & \text{si } X = \text{categoría } j \\
0 & \text{en otro caso}
\end{cases}
$$

para $j = 1, \dots, k$.

A diferencia de las variables dummy, no elimina ninguna categoría, por lo que puede generar multicolinealidad si se usa en modelos lineales sin regularización.

---

### **Diferencias**

- Variables Dummy: generan $k-1$ variables y evitan multicolinealidad.

- One-Hot Encoding: generan $k$ variables y conservan toda la información.

- Dummy es preferido en modelos lineales clásicos.

- One-Hot es común en pipelines de Machine Learning.

---
<p align="center">
   <img src="img/dumi.png" width="900" height="500">
</p>

---

# Escalamiento y Estandarización 

En Analítica de Datos, las variables pueden estar en escalas muy diferentes:

- Edad → 18 a 70

- Ingreso → 1,000 a 10,000,000

- Horas → 0 a 60

- Razón → 0 a 1

👉 Esto puede sesgar modelos y métricas.

---

### ¿Qué es escalar variables?

Es transformar los datos para que estén en una misma escala, sin alterar su información esencial.

---

### 1. Estandarización

Es el proceso de aplicar a una (variable )columna del conjunto de datos, la siguiente transformación:

$$z = \frac{x - \mu}{\sigma}$$

Interpretación:

- Centra los datos en media 0

- Desviación estándar = 1

---

### 2. Normalización (Min-Max Scaling)

Es el proceso de aplicar a una (variable )columna del conjunto de datos, la siguiente transformación:

$$w = \frac{x - x_{mín}}{x_{máx}-x_{mín}}$$

Interpretación:

- Escala entre 0 y 1

- Mantiene proporciones

---

### Diferencias:

| Método         | Sensibilidad a outliers | Rango      |
| -------------- | ----------------------- | ---------- |
| StandardScaler | Baja                    | No acotado |
| MinMaxScaler   | Alta                    | [0,1]      |

---

### ¿Por qué se hace?

- Evita que variables dominen el modelo

- Mejora convergencia numérica

- Hace comparables las variables

- Mejora rendimiento en ML

---

### ¿Cuándo es necesario?

Hay Algoritmos sensibles a escala, tales como:

- KNN (distancias)

- SVM

- Regresión con regularización (Ridge, Lasso)

- Redes neuronales

- PCA

---

### ¿Cuándo NO es necesario?

- Árboles de decisión 

- Random Forest

- XGBoost

"Porque no usan distancias ni magnitudes"

---

<div align="center">
  <span style="font-size: 2em;">"Los modelos que usan distancias necesitan escalamiento”</span>
</div>

---

En algoritmos basados en distancias (como KNN, K-Means o SVM), las variables con escalas numéricas grandes dominan a las pequeñas, creando un sesgo artificial.

Consideremos un conjunto de datos con dos variables de escalas muy distintas: 

- **Edad** (rango aprox. 18-90)  

- **Ingresos Anuales** (rango aprox. 20,000-200,000). 

Definimos tres clientes como vectores en $\mathbb{R}^2$: 

$$P=[Edad, Ingresos]$$


| Cliente        | Edad ($x_1$) | Ingresos ($x_2$) |
|----------------|-------------|------------------|
| Cliente A ($P_1$) | 25          | $50,000          |
| Cliente B ($P_2$) | 26          | $55,000          |
| Cliente C ($P_3$) | 80          | $50,100          |

---

**Cálculo de Distancia Euclidiana**

La distancia entre dos puntos $P$ y $Q$ se calcula mediante:

$$d(P, Q) = \sqrt{(q_1 - p_1)^2 + (q_2 - p_2)^2}$$

---

**Distancia entre Cliente A y Cliente B**

\begin{align*}
d(P_1, P_2) &= \sqrt{(26 - 25)^2 + (55,000 - 50,000)^2} \\
d(P_1, P_2) &= \sqrt{1^2 + 5,000^2} \\
d(P_1, P_2) &= \sqrt{25,000,001} \approx 5,000.00
\end{align*}

---

**Distancia entre Cliente A y Cliente C**

\begin{align*}
d(P_1, P_3) &= \sqrt{(80 - 25)^2 + (50,100 - 50,000)^2} \\
d(P_1, P_3) &= \sqrt{55^2 + 100^2} \\
d(P_1, P_3) &= \sqrt{3,025 + 10,000} \\
d(P_1, P_3) &= \sqrt{13,025} \approx 114.12
\end{align*}

---

**Conclusión**

Dado que $d(P_1, P_3) < d(P_1, P_2)$, un algoritmo como K-Nearest Neighbors (KNN) clasificaría erróneamente al Cliente C como más similar al Cliente A. Esto ocurre porque la diferencia de $5,000 en ingresos **domina** matemáticamente sobre la diferencia de 55 años de edad.


Para resolver esto, es imperativo aplicar una transformación de \textbf{Estandarización (Z-score)}:

$$z = \frac{x - \mu}{\sigma}$$

Donde cada característica será reescalada para tener media $\mu = 0$ y desviación estándar $\sigma = 1$.

| Par de Puntos     | Distancia Sin Estandarizar | Distancia Estandarizada | ¿Qué cambió? |
|:------------------|--------------------------:|-----------------------:|:-------------|
| $d(P_1, P_2)$     | 5,000.00                  | 1.74                   | Antes eran los más "lejanos", ahora están a una distancia moderada.|
| $d(P_1, P_3)$     | 114.13                    | 1.74                   | Antes eran los más "cercanos", ahora valen lo mismo que $P_1 - P_2$. |
| $d(P_2, P_3)$     | 4,900.30                  | 2.42                   | Se confirma como la distancia real más grande entre perfiles. |

---

# ¡FIN!

---